# Signature-transform forecasting on public labor-market data

This notebook applies the signature-transform forecasting framework to a fully public weekly dataset built from FRED. The target is **U.S. initial jobless claims** and the covariates are macro and market series that are available at daily or weekly frequency. The goal is to test the same adaptive signature approach on a second dataset with a different economic interpretation.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd

from marketplace_signature_forecast.config import (
    ALL_HORIZONS,
    DEFAULT_DEPTH,
    DEFAULT_N_TARGET,
    DEFAULT_WINDOW_SIZE,
    FRED_API_KEY,
)
from marketplace_signature_forecast.data_loading import (
    build_data_dictionary_from_specs,
    fetch_series_bundle,
)
from marketplace_signature_forecast.dataset_specs import (
    INITIAL_CLAIMS_END_DATE,
    INITIAL_CLAIMS_EXPERIMENT_NAME,
    INITIAL_CLAIMS_FRED_INPUTS,
    INITIAL_CLAIMS_N_VALIDATION,
    INITIAL_CLAIMS_PLOT_HORIZON,
    INITIAL_CLAIMS_RESAMPLE_FREQ,
    INITIAL_CLAIMS_START_DATE,
    INITIAL_CLAIMS_TARGET,
    INITIAL_CLAIMS_YAHOO_INPUTS,
)
from marketplace_signature_forecast.preprocessing import (
    build_model_dataset,
    prepare_standardized_arrays,
    resample_collection,
    resample_to_weekly,
    save_processed_bundle,
    train_validation_split,
)
from marketplace_signature_forecast.adaptive_weights import rolling_adaptive_weights
from marketplace_signature_forecast.evaluation import (
    build_forecast_dataframe,
    compare_with_baselines,
    inverse_transform_forecasts,
    run_multi_horizon_experiment,
)
from marketplace_signature_forecast.modeling import rolling_forecast
from marketplace_signature_forecast.plotting import (
    plot_adaptive_weight_diagnostics,
    plot_correlation_matrix,
    plot_forecast_vs_actual,
    plot_input_grid,
    plot_target_split,
    plot_weights_for_origin,
)

DATASET_DIR = PROJECT_ROOT / "data" / "processed" / INITIAL_CLAIMS_EXPERIMENT_NAME
FIGURE_DIR = PROJECT_ROOT / "figures" / INITIAL_CLAIMS_EXPERIMENT_NAME
Y_ONLY_DIR = DATASET_DIR / "signature_y_only"
JOINT_DIR = DATASET_DIR / "signature_joint_path"

for directory in [DATASET_DIR, FIGURE_DIR, Y_ONLY_DIR, JOINT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

data_dictionary = build_data_dictionary_from_specs(
    target=INITIAL_CLAIMS_TARGET,
    fred_inputs=INITIAL_CLAIMS_FRED_INPUTS,
    yahoo_inputs=INITIAL_CLAIMS_YAHOO_INPUTS,
)
data_dictionary

## 1. Download, align, and save the weekly dataset

All series are resampled to the same weekly calendar ending on Saturday so that the target and covariates share a common forecasting grid.

In [ ]:
y_raw, x_raw = fetch_series_bundle(
    api_key=FRED_API_KEY,
    target=INITIAL_CLAIMS_TARGET,
    start=INITIAL_CLAIMS_START_DATE,
    end=INITIAL_CLAIMS_END_DATE,
    fred_inputs=INITIAL_CLAIMS_FRED_INPUTS,
    yahoo_inputs=INITIAL_CLAIMS_YAHOO_INPUTS,
)

y_weekly = resample_to_weekly(y_raw, freq=INITIAL_CLAIMS_RESAMPLE_FREQ)
x_weekly = resample_collection(x_raw, freq=INITIAL_CLAIMS_RESAMPLE_FREQ)
full_data = build_model_dataset(y_weekly, x_weekly).dropna().copy()
train_data, validation_data = train_validation_split(full_data, INITIAL_CLAIMS_N_VALIDATION)

save_processed_bundle(full_data, train_data, validation_data, data_dictionary, DATASET_DIR)

print(full_data.shape)
print(full_data.index.min(), full_data.index.max())
full_data.head()

## 2. Exploratory analysis

In [ ]:
plot_target_split(train_data, validation_data, INITIAL_CLAIMS_TARGET["name"], FIGURE_DIR / "target_variable.png")
plot_input_grid(full_data, INITIAL_CLAIMS_TARGET["name"], FIGURE_DIR / "input_variables.png")
plot_correlation_matrix(full_data, FIGURE_DIR / "correlation_matrix.png")

full_data.describe().round(2)

## 3. Standardized modeling arrays

In [ ]:
prepared = prepare_standardized_arrays(
    full_data=full_data,
    target_col=INITIAL_CLAIMS_TARGET["name"],
    horizons=ALL_HORIZONS,
    n_validation=INITIAL_CLAIMS_N_VALIDATION,
)

X = prepared["X"]
y = prepared["y"]
y_raw_array = prepared["y_raw"]
dates = prepared["dates"]
y_scaler = prepared["y_scaler"]
train_end_t = prepared["train_end"]

WINDOW_SIZE = DEFAULT_WINDOW_SIZE
DEPTH = DEFAULT_DEPTH
N_TARGET = DEFAULT_N_TARGET

print(X.shape, y.shape, train_end_t)

## 4. Adaptive-weight diagnostics

In [ ]:
diagnostic_horizon = INITIAL_CLAIMS_PLOT_HORIZON
min_t = WINDOW_SIZE + diagnostic_horizon + 20

adaptive_results = rolling_adaptive_weights(
    X=X,
    y=y,
    start_t=min_t,
    end_t=train_end_t,
    delta_t=diagnostic_horizon,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    gamma=None,
    n_target=N_TARGET,
    add_time=True,
)

plot_adaptive_weight_diagnostics(adaptive_results, dates, y, train_end_t, N_TARGET, INITIAL_CLAIMS_TARGET["name"])
plot_weights_for_origin(adaptive_results, len(adaptive_results["t"]) // 2, X, y, dates, diagnostic_horizon, WINDOW_SIZE, DEPTH)

## 5. Multi-horizon point forecasts: target-path signature only

In [ ]:
results_y_only, summary_y_only = run_multi_horizon_experiment(
    X=X,
    y=y,
    dates=dates,
    y_scaler=y_scaler,
    horizons=ALL_HORIZONS,
    n_validation=INITIAL_CLAIMS_N_VALIDATION,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    n_target=N_TARGET,
    output_dir=Y_ONLY_DIR,
    lambda_lasso=None,
    gamma=None,
    use_sig_y_only=True,
    add_time=True,
)

summary_y_only = summary_y_only.rename(columns={"signature_mre": "signature_y_only_mre"})
summary_y_only

## 6. Multi-horizon point forecasts: joint-path signature

In [ ]:
results_joint, summary_joint = run_multi_horizon_experiment(
    X=X,
    y=y,
    dates=dates,
    y_scaler=y_scaler,
    horizons=ALL_HORIZONS,
    n_validation=INITIAL_CLAIMS_N_VALIDATION,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    n_target=N_TARGET,
    output_dir=JOINT_DIR,
    lambda_lasso=None,
    gamma=None,
    use_sig_y_only=False,
    add_time=True,
)

summary_joint = summary_joint.rename(columns={"signature_mre": "signature_joint_path_mre"})
summary_joint

## 7. Baseline comparison

In [ ]:
comparison_y_only = compare_with_baselines(
    y_raw=y_raw_array,
    horizons=ALL_HORIZONS,
    n_validation=INITIAL_CLAIMS_N_VALIDATION,
    signature_summary=summary_y_only.rename(columns={"signature_y_only_mre": "signature_mre"}),
).rename(
    columns={
        "Signature (%)": "Signature y-only (%)",
        "Improvement vs ARIMA (%)": "Improvement y-only vs ARIMA (%)",
    }
)

comparison_table = comparison_y_only.merge(
    summary_joint[["horizon_weeks", "signature_joint_path_mre"]],
    left_on="Horizon (weeks)",
    right_on="horizon_weeks",
    how="left",
).drop(columns=["horizon_weeks"])

comparison_table = comparison_table.rename(columns={"signature_joint_path_mre": "Signature joint-path (%)"})
comparison_table["Improvement joint-path vs ARIMA (%)"] = (
    (comparison_table["ARIMA (%)"] - comparison_table["Signature joint-path (%)"])
    / comparison_table["ARIMA (%)"] * 100
)

comparison_table.to_csv(DATASET_DIR / "comparison_initial_claims.csv", index=False)
comparison_table.round(2)

## 8. Forecast paths for a representative horizon

In [ ]:
delta_t = INITIAL_CLAIMS_PLOT_HORIZON
val_start_t = len(y) - INITIAL_CLAIMS_N_VALIDATION - delta_t
val_end_t = len(y) - delta_t - 1

forecast_y_only = rolling_forecast(
    X=X,
    y=y,
    start_t=val_start_t,
    end_t=val_end_t,
    delta_t=delta_t,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    lambda_lasso=None,
    gamma=None,
    n_target=N_TARGET,
    use_sig_y_only=True,
    add_time=True,
)

forecast_joint = rolling_forecast(
    X=X,
    y=y,
    start_t=val_start_t,
    end_t=val_end_t,
    delta_t=delta_t,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    lambda_lasso=None,
    gamma=None,
    n_target=N_TARGET,
    use_sig_y_only=False,
    add_time=True,
)

metrics_y_only = inverse_transform_forecasts(forecast_y_only, y_scaler)
metrics_joint = inverse_transform_forecasts(forecast_joint, y_scaler)

df_y_only = build_forecast_dataframe(forecast_y_only, dates, delta_t, metrics_y_only)
df_joint = build_forecast_dataframe(forecast_joint, dates, delta_t, metrics_joint)

plot_forecast_vs_actual(df_y_only, delta_t)
plot_forecast_vs_actual(df_joint, delta_t)

plot_df = df_y_only[["target_date", "actual_orig"]].merge(
    df_y_only[["target_date", "forecast_orig"]].rename(columns={"forecast_orig": "signature_y_only"}),
    on="target_date",
    how="left",
).merge(
    df_joint[["target_date", "forecast_orig"]].rename(columns={"forecast_orig": "signature_joint"}),
    on="target_date",
    how="left",
)

plot_df.to_csv(DATASET_DIR / f"forecast_path_comparison_delta{delta_t}.csv", index=False)
plot_df.head()